# 🎙️ GOLDTECH Voice — free GPU voice cloning

Clone **your own voice** and read any script on a free GPU.
Powered by **Chatterbox Turbo** (MIT-licensed, by Resemble AI).

**Works on:** Google Colab · Kaggle · Lightning AI Studio · SageMaker Studio Lab · any CUDA machine.
The notebook auto-detects the platform.

## How to run (once per session)
1. Turn on the GPU:
   - **Colab:** Runtime → Change runtime type → **T4 GPU**.
   - **Kaggle:** Settings → Accelerator → **GPU T4 x2** (or P100), and turn **Internet ON**.
   - **Lightning / SageMaker:** pick a GPU instance/runtime.
2. **If you ran an older version of this notebook, restart the session first**
   (Kaggle: Run → Restart Session and Clear Output; Colab: Runtime → Restart session).
3. **Run all** cells top to bottom.
4. (Colab only) Approve the Google Drive prompt to cache the model — optional.
5. Copy the **`https://….trycloudflare.com`** URL printed by the last cell.
6. Open the **GOLDTECH Voice** web app → **Connect engine** → paste the URL.

> The URL changes each session. Keep the notebook tab open while generating.
> If a session dies, re-run the last cell for a fresh URL.

**Only clone a voice you own or have explicit permission to use.**


In [ ]:
# @title 1) Check GPU + install engine (Kaggle-safe)
import sys, subprocess, shutil, os, sysconfig, urllib.request, importlib.metadata
from pathlib import Path
import os
def _gt_base():
    if os.path.isdir("/kaggle/working"):      # Kaggle
        return "/kaggle/working/gt_voice"
    if os.path.isdir("/content"):             # Google Colab
        return "/content/gt_voice"
    return os.path.abspath("gt_voice")        # Lightning / SageMaker / local
BASE = _gt_base()
os.makedirs(BASE, exist_ok=True)
os.environ["GT_VOICE_BASE"] = BASE
def _gt_platform():
    if os.path.isdir("/kaggle/working"): return "Kaggle"
    if os.path.isdir("/content"): return "Google Colab"
    return "GPU host"

print("Platform:", _gt_platform(), "| work dir:", BASE)

# Kaggle's system Python can be PEP 668 "externally managed": pip refuses to
# install unless we pass --break-system-packages. Detect the marker and add it.
_marker = os.path.join(sysconfig.get_path("stdlib"), "EXTERNALLY-MANAGED")
_PIP_FLAGS = ["--break-system-packages"] if os.path.exists(_marker) else []

def pip(*args):
    base = [sys.executable, "-m", "pip", "install", "--no-input"] + _PIP_FLAGS
    r = subprocess.run(base + list(args), capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0 and "externally-managed" in out.lower() and "--break-system-packages" not in base:
        r = subprocess.run(base + ["--break-system-packages"] + list(args), capture_output=True, text=True)
        out = (r.stdout or "") + (r.stderr or "")
    if r.returncode != 0:
        print(out[-4000:])   # show the REAL pip error, not a bare exit code
        raise RuntimeError(f"pip install failed ({r.returncode}): {' '.join(args)}")
    tail = [l for l in out.strip().splitlines() if l.strip()][-2:]
    print("\n".join(tail))

# Quick internet check — most Kaggle pip failures are simply Internet = OFF.
try:
    urllib.request.urlopen("https://pypi.org", timeout=10)
    print("Internet: OK (PyPI reachable)")
except Exception:
    print("⚠️ Cannot reach PyPI. Turn ON Internet in the Kaggle right-sidebar Settings, then re-run.")

# Check the GPU in a FRESH interpreter. We deliberately do NOT import torch or
# numpy into the notebook kernel here: changing packages after they are loaded
# is what caused the earlier numpy.ufunc / LlamaModel import crash.
_gpu_code = "import torch, json; print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}))"
r = subprocess.run([sys.executable, "-c", _gpu_code], capture_output=True, text=True)
if r.returncode == 0:
    import json as _json
    info = _json.loads(r.stdout.strip().splitlines()[-1])
    print("PyTorch:", info["torch"], "| CUDA available:", info["cuda"])
    if info["cuda"]:
        print("GPU:", info["gpu"])
    else:
        print("⚠️ No GPU detected. Enable a GPU accelerator in notebook settings. Continuing on CPU (very slow).")
else:
    print("⚠️ Could not check torch:", ((r.stderr or "") + (r.stdout or ""))[-800:])

# Chatterbox pins torch==2.6.0 which would replace the host's CUDA build.
# Install the prebuilt PyPI wheel with --no-deps (no git/build step), keeping the
# preinstalled GPU torch. Then add only the real runtime dependencies.
print("\nInstalling Chatterbox (prebuilt wheel, no-deps, keeps your GPU torch)…")
pip("--no-deps", "chatterbox-tts==0.1.7")

# Lock the host's preinstalled scientific/CUDA stack so pip cannot downgrade or
# replace it. This is the key fix for the numpy.ufunc '__module__' crash: we no
# longer force numpy==2.2.6 over Kaggle's newer numpy.
constraints = []
for pkg in ["numpy", "scipy", "scikit-learn", "numba", "llvmlite", "torch", "torchaudio"]:
    try:
        constraints.append(f"{pkg}=={importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        pass
cfile = Path(BASE) / "constraints.txt"
cfile.write_text("\n".join(constraints) + "\n")
print("Locked host packages:", ", ".join(constraints) or "(none found)")

print("Installing runtime dependencies…")
pip("-c", str(cfile),
    "transformers==5.2.0", "diffusers==0.29.0", "librosa==0.11.0", "s3tokenizer",
    "conformer==0.3.2", "resemble-perth", "pyloudnorm", "omegaconf", "einops",
    "onnxruntime", "soundfile", "fastapi", "uvicorn", "python-multipart",
    "imageio-ffmpeg", "tqdm")

# Verify every import in a FRESH interpreter before we ever load the model.
_smoke = "import numpy, scipy.sparse, sklearn, torch, torchaudio, transformers, librosa\nimport soundfile, pyloudnorm, omegaconf, einops, onnxruntime, fastapi, uvicorn, tqdm\nfrom transformers import LlamaModel, LlamaConfig, GPT2Config, GPT2Model, GenerationMixin\nfrom chatterbox.tts_turbo import ChatterboxTurboTTS\nprint('SMOKE OK | numpy', numpy.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)\n"
r = subprocess.run([sys.executable, "-c", _smoke], capture_output=True, text=True)
if r.returncode != 0:
    print(((r.stdout or "") + (r.stderr or ""))[-4000:])
    raise RuntimeError("Import smoke test failed — see the real error above.")
print(r.stdout.strip())

# ffmpeg: prefer system binary; otherwise the imageio-ffmpeg wheel provides one.
if not shutil.which("ffmpeg"):
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
    except Exception:
        print("apt-get unavailable — using imageio-ffmpeg bundled binary.")

# cloudflared gives us a free public HTTPS URL for the engine.
CF = os.path.join(BASE, "cloudflared")
if not os.path.exists(CF):
    print("Downloading cloudflared…")
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", CF], check=True)
    os.chmod(CF, 0o755)
print("\n✅ Setup complete.")


In [ ]:
# @title 2) (Optional) Cache the model for faster restarts
import os
import os
def _gt_base():
    if os.path.isdir("/kaggle/working"):      # Kaggle
        return "/kaggle/working/gt_voice"
    if os.path.isdir("/content"):             # Google Colab
        return "/content/gt_voice"
    return os.path.abspath("gt_voice")        # Lightning / SageMaker / local
BASE = _gt_base()
os.makedirs(BASE, exist_ok=True)
os.environ["GT_VOICE_BASE"] = BASE
def _gt_platform():
    if os.path.isdir("/kaggle/working"): return "Kaggle"
    if os.path.isdir("/content"): return "Google Colab"
    return "GPU host"

plat = _gt_platform()
if plat == "Google Colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        cache = "/content/drive/MyDrive/GOLDTECH-Voice/hf_cache"
        os.makedirs(cache, exist_ok=True)
        os.environ["HF_HOME"] = cache
        print("✅ Model cache →", cache)
    except Exception as e:
        print("Drive not mounted (", e, "). Model re-downloads each session.")
elif plat == "Kaggle":
    cache = os.path.join(BASE, "hf_cache")
    os.makedirs(cache, exist_ok=True)
    os.environ["HF_HOME"] = cache
    print("✅ Model cache →", cache, "(persists in this session's working dir)")
else:
    cache = os.path.join(BASE, "hf_cache")
    os.makedirs(cache, exist_ok=True)
    os.environ["HF_HOME"] = cache
    print("✅ Model cache →", cache)


## 3) Write the engine server
Run this cell — it writes `server.py` (the FastAPI engine) into the working dir. Expected output: `Writing gt_voice/server.py`.

In [ ]:
%%writefile gt_voice/server.py
# GOLDTECH Voice — Colab engine server
# Runs inside Google Colab. Loads Chatterbox Turbo once, keeps it warm on the GPU,
# and exposes a small JSON API that the GOLDTECH Voice web app talks to through a
# Cloudflare tunnel. One job at a time; progress is polled.
#
# This file is written to /content/gt_voice/server.py by the notebook, then imported.

import os
import re
import sys
import time
import uuid
import math
import shutil
import threading
import subprocess
from pathlib import Path

import numpy as np

BASE = Path(os.environ.get("GT_VOICE_BASE", "/content/gt_voice"))
JOBS = BASE / "jobs"
JOBS.mkdir(parents=True, exist_ok=True)

# Filled in by the notebook's run cell after the model loads.
MODEL = None
DEVICE = "cpu"
SR = 24000
MODEL_NAME = "chatterbox-turbo"

MAX_SCRIPT_CHARS = 20000
REF_MIN_SEC = 6.0      # model asserts > 5s; we require a little more for accuracy
REF_MAX_SEC = 30.0     # longer references don't help and slow conditioning
TARGET_LUFS = -16.0    # loudness of the finished voiceover

_jobs = {}
_jobs_lock = threading.Lock()
_gpu_lock = threading.Lock()   # serialize GPU work


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def _ffmpeg():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    # Fallback for Kaggle / Studio environments where apt-get isn't available.
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None


def convert_to_wav(src: Path, dst: Path, sr: int = 24000):
    """Decode any uploaded audio (webm/opus/mp3/m4a/wav…) to mono PCM WAV."""
    ff = _ffmpeg()
    if not ff:
        raise RuntimeError("ffmpeg is not installed in this runtime")
    cmd = [ff, "-y", "-hide_banner", "-loglevel", "error", "-i", str(src),
           "-ac", "1", "-ar", str(sr), "-c:a", "pcm_s16le", str(dst)]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0 or not dst.exists():
        raise RuntimeError(f"Could not decode audio: {p.stderr.strip()[:300]}")


def prepare_reference(src: Path, dst: Path) -> dict:
    """Convert, trim silence, normalize, and length-check the reference clip."""
    import librosa
    import soundfile as sf

    tmp = dst.with_suffix(".raw.wav")
    convert_to_wav(src, tmp, sr=SR)
    y, _ = librosa.load(str(tmp), sr=SR, mono=True)
    try:
        tmp.unlink()
    except OSError:
        pass

    if y.size == 0:
        raise ValueError("Reference audio is empty")

    # Trim leading/trailing silence so the model hears speech, not room tone.
    yt, _ = librosa.effects.trim(y, top_db=30, frame_length=1024, hop_length=256)
    if yt.size < SR * 1.0:
        yt = y  # trimming was too aggressive; keep the original

    dur = len(yt) / SR
    if dur < REF_MIN_SEC:
        raise ValueError(
            f"Reference is only {dur:.1f}s after trimming silence. "
            f"Record at least {REF_MIN_SEC:.0f}s of clear speech (15–20s is ideal)."
        )
    if dur > REF_MAX_SEC:
        yt = yt[: int(REF_MAX_SEC * SR)]
        dur = REF_MAX_SEC

    # Loudness-normalize to the model's expected -27 LUFS ourselves, keeping the
    # array strictly float32. Chatterbox's internal norm_loudness can promote the
    # waveform to float64 (pyloudnorm returns a numpy.float64 gain), which causes
    # "expected scalar type Float but found Double" on CUDA. We disable that
    # internal step below and hand the model a clean float32 reference instead.
    normalized = False
    try:
        import pyloudnorm as ln
        meter = ln.Meter(SR)
        loud = meter.integrated_loudness(yt)
        if math.isfinite(loud):
            gain = np.float32(10.0 ** ((-27.0 - float(loud)) / 20.0))
            yt = yt * gain
            normalized = True
    except Exception:
        pass

    peak = float(np.max(np.abs(yt))) or 1.0
    if not normalized:
        yt = yt * np.float32(0.891 / peak)   # fallback: peak-normalize to -1 dBFS
    elif peak > 0.99:
        yt = yt * np.float32(0.99 / peak)    # safety limiter only

    yt = np.ascontiguousarray(yt, dtype=np.float32)
    sf.write(str(dst), yt, SR, subtype="PCM_16")
    return {"seconds": round(dur, 2), "sample_rate": SR}


def chunk_text(text: str, max_chars: int):
    """Split a script into <= max_chars chunks on sentence/phrase boundaries."""
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, cur = [], ""

    def push(piece):
        nonlocal cur
        piece = piece.strip()
        if not piece:
            return
        if cur and len(cur) + 1 + len(piece) <= max_chars:
            cur = (cur + " " + piece).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = piece

    for s in sentences:
        if len(s) <= max_chars:
            push(s)
            continue
        # Long sentence: break on commas/semicolons/colons, then hard-wrap words.
        for part in re.split(r"(?<=[,;:])\s+", s):
            if len(part) <= max_chars:
                push(part)
                continue
            words, buf = part.split(), ""
            for w in words:
                if buf and len(buf) + 1 + len(w) > max_chars:
                    push(buf)
                    buf = w
                else:
                    buf = (buf + " " + w).strip()
            if buf:
                push(buf)
    if cur:
        chunks.append(cur)
    return chunks


def _fade(y: np.ndarray, ms: float = 6.0) -> np.ndarray:
    n = min(int(SR * ms / 1000.0), len(y) // 2)
    if n <= 0:
        return y
    ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
    y = y.copy()
    y[:n] *= ramp
    y[-n:] *= ramp[::-1]
    return y


def _normalize_loudness(y: np.ndarray) -> np.ndarray:
    try:
        import pyloudnorm as ln
        meter = ln.Meter(SR)
        loud = meter.integrated_loudness(y)
        if math.isfinite(loud):
            gain = 10.0 ** ((TARGET_LUFS - loud) / 20.0)
            y = y * gain
    except Exception:
        pass
    peak = float(np.max(np.abs(y))) or 1.0
    if peak > 0.99:
        y = y * (0.99 / peak)
    return y.astype(np.float32)


def _float32_conditionals(conds):
    """Force any floating conditioning tensors to float32 (keep int tokens intact)."""
    import torch
    if conds is None:
        return
    t3 = getattr(conds, "t3", None)
    if t3 is not None:
        for k, v in list(vars(t3).items()):
            if torch.is_tensor(v) and v.is_floating_point() and v.dtype != torch.float32:
                setattr(t3, k, v.to(torch.float32))
    gen = getattr(conds, "gen", None)
    if isinstance(gen, dict):
        for k, v in list(gen.items()):
            if torch.is_tensor(v) and v.is_floating_point() and v.dtype != torch.float32:
                gen[k] = v.to(torch.float32)


# --------------------------------------------------------------------------- #
# Job runner
# --------------------------------------------------------------------------- #
def _run_job(job_id: str):
    import soundfile as sf
    import torch

    with _jobs_lock:
        job = _jobs[job_id]
    root = JOBS / job_id
    ref_path = root / "reference.wav"
    out_wav = root / "voiceover.wav"
    out_mp3 = root / "voiceover.mp3"

    try:
        with _jobs_lock:
            job["status"] = "running"
            job["message"] = "Preparing reference voice…"

        ref_info = prepare_reference(root / job["ref_upload_name"], ref_path)
        with _jobs_lock:
            job["reference"] = ref_info

        chunks = chunk_text(job["script"], job["max_chars"])
        if not chunks:
            raise ValueError("Script is empty")
        with _jobs_lock:
            job["chunks_total"] = len(chunks)
            job["message"] = f"Cloning voice… 0/{len(chunks)}"

        if job["seed"] is not None:
            torch.manual_seed(int(job["seed"]))

        segments = []
        pause_samples = int(SR * job["pause"])

        # Serialize GPU access; keep the model warm between jobs.
        with _gpu_lock:
            if MODEL is None:
                raise RuntimeError("Model is not loaded yet")
            # Condition once on the reference, then reuse for every chunk so the
            # voice stays identical across the whole script. norm_loudness=False:
            # we already loudness-normalized the reference to -27 LUFS in float32.
            MODEL.prepare_conditionals(str(ref_path), exaggeration=0.0, norm_loudness=False)
            _float32_conditionals(getattr(MODEL, "conds", None))

            for i, text in enumerate(chunks):
                if job.get("cancel"):
                    raise RuntimeError("Cancelled")
                with _jobs_lock:
                    job["message"] = f"Cloning voice… {i}/{len(chunks)}"
                wav = MODEL.generate(
                    text,
                    temperature=job["temperature"],
                    repetition_penalty=1.2,
                    top_p=0.95,
                    norm_loudness=False,
                )
                _float32_conditionals(getattr(MODEL, "conds", None))
                y = wav.detach().to(torch.float32).squeeze(0).cpu().numpy()
                if y.size < SR * 0.2:
                    continue  # skip a degenerate chunk
                segments.append(_fade(y))
                if i < len(chunks) - 1 and pause_samples > 0:
                    segments.append(np.zeros(pause_samples, dtype=np.float32))
                with _jobs_lock:
                    job["chunks_done"] = i + 1
                    job["progress"] = round((i + 1) / len(chunks), 4)
                    job["message"] = f"Cloning voice… {i + 1}/{len(chunks)}"

        if not segments:
            raise RuntimeError("Generation produced no audio")

        with _jobs_lock:
            job["message"] = "Mixing and normalizing…"
        audio = _normalize_loudness(np.concatenate(segments))
        sf.write(str(out_wav), audio, SR, subtype="PCM_16")

        if _ffmpeg():
            p = subprocess.run(
                [_ffmpeg(), "-y", "-hide_banner", "-loglevel", "error",
                 "-i", str(out_wav), "-codec:a", "libmp3lame", "-qscale:a", "2", str(out_mp3)],
                capture_output=True, text=True,
            )
            if p.returncode != 0:
                out_mp3 = None
        else:
            out_mp3 = None

        with _jobs_lock:
            job["status"] = "done"
            job["progress"] = 1.0
            job["message"] = "Voiceover ready"
            job["duration"] = round(len(audio) / SR, 2)
            job["audio"] = str(out_wav)
            job["mp3"] = str(out_mp3) if out_mp3 and out_mp3.exists() else None
            job["finished"] = time.time()

    except Exception as e:
        with _jobs_lock:
            cancelled = job.get("cancel")
            job["status"] = "cancelled" if cancelled else "error"
            job["message"] = "Cancelled" if cancelled else str(e)
            job["error"] = None if cancelled else str(e)
            job["finished"] = time.time()


# --------------------------------------------------------------------------- #
# Web API
# --------------------------------------------------------------------------- #
def create_app():
    from fastapi import FastAPI, File, Form, UploadFile, HTTPException
    from fastapi.responses import FileResponse, JSONResponse, HTMLResponse
    from fastapi.middleware.cors import CORSMiddleware

    app = FastAPI(title="GOLDTECH Voice Engine")
    app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],          # personal ephemeral tunnel; no credentials
        allow_methods=["*"],
        allow_headers=["*"],
    )

    @app.get("/", response_class=HTMLResponse)
    def index():
        return ("<html><body style='font-family:sans;padding:2rem'>"
                "<h2>GOLDTECH Voice engine is running.</h2>"
                "<p>Open the GOLDTECH Voice web app and paste this tunnel URL.</p>"
                "</body></html>")

    @app.get("/api/health")
    def health():
        return {
            "ok": True,
            "model": MODEL_NAME,
            "device": DEVICE,
            "sample_rate": SR,
            "warm": MODEL is not None,
            "gpu": DEVICE == "cuda",
        }

    @app.post("/api/jobs")
    async def create_job(
        script: str = Form(...),
        reference: UploadFile = File(...),
        temperature: float = Form(0.8),
        pause: float = Form(0.35),
        max_chars: int = Form(220),
        seed: str = Form(""),
        consent: str = Form("false"),
    ):
        if consent != "true":
            raise HTTPException(400, "You must confirm you own the voice or have permission to clone it.")
        if MODEL is None:
            raise HTTPException(503, "Model is still loading. Wait a moment and try again.")
        script = script.strip()
        if not script:
            raise HTTPException(400, "Script is empty.")
        if len(script) > MAX_SCRIPT_CHARS:
            raise HTTPException(400, f"Script too long (max {MAX_SCRIPT_CHARS} characters).")
        if not (0.1 <= temperature <= 1.5):
            raise HTTPException(400, "Temperature must be 0.1–1.5.")
        if not (0.0 <= pause <= 2.0):
            raise HTTPException(400, "Pause must be 0–2 seconds.")
        if not (60 <= max_chars <= 500):
            raise HTTPException(400, "Chunk size must be 60–500.")
        seed_val = None
        if seed.strip():
            try:
                seed_val = int(seed.strip())
            except ValueError:
                raise HTTPException(400, "Seed must be a whole number.")

        with _jobs_lock:
            running = [j for j in _jobs.values() if j["status"] in ("queued", "running")]
            if running:
                raise HTTPException(409, "A job is already running. Wait for it to finish or cancel it.")

        job_id = uuid.uuid4().hex[:12]
        root = JOBS / job_id
        root.mkdir(parents=True, exist_ok=True)
        upload_name = "reference" + Path(reference.filename or "ref.wav").suffix.lower()
        if upload_name == "reference":
            upload_name = "reference.webm"
        dest = root / upload_name
        data = await reference.read()
        if len(data) > 40 * 1024 * 1024:
            raise HTTPException(400, "Reference audio too large (max 40 MB).")
        dest.write_bytes(data)

        job = {
            "id": job_id,
            "status": "queued",
            "progress": 0.0,
            "message": "Queued",
            "script": script,
            "ref_upload_name": upload_name,
            "temperature": float(temperature),
            "pause": float(pause),
            "max_chars": int(max_chars),
            "seed": seed_val,
            "chunks_total": 0,
            "chunks_done": 0,
            "created": time.time(),
            "cancel": False,
        }
        with _jobs_lock:
            _jobs[job_id] = job
        threading.Thread(target=_run_job, args=(job_id,), daemon=True).start()
        return {"job_id": job_id}

    @app.get("/api/jobs/{job_id}")
    def job_status(job_id: str):
        with _jobs_lock:
            job = _jobs.get(job_id)
            if not job:
                raise HTTPException(404, "Job not found")
            return {k: v for k, v in job.items()
                    if k not in ("script", "ref_upload_name", "cancel")}

    @app.post("/api/jobs/{job_id}/cancel")
    def cancel_job(job_id: str):
        with _jobs_lock:
            job = _jobs.get(job_id)
            if not job:
                raise HTTPException(404, "Job not found")
            job["cancel"] = True
        return {"ok": True}

    @app.get("/api/jobs/{job_id}/audio")
    def job_audio(job_id: str):
        with _jobs_lock:
            job = _jobs.get(job_id)
        if not job or job["status"] != "done" or not job.get("audio"):
            raise HTTPException(404, "Audio not ready")
        return FileResponse(job["audio"], media_type="audio/wav", filename="goldtech-voiceover.wav")

    @app.get("/api/jobs/{job_id}/mp3")
    def job_mp3(job_id: str):
        with _jobs_lock:
            job = _jobs.get(job_id)
        if not job or job["status"] != "done" or not job.get("mp3"):
            raise HTTPException(404, "MP3 not ready")
        return FileResponse(job["mp3"], media_type="audio/mpeg", filename="goldtech-voiceover.mp3")

    return app


app = create_app()


## 4) Write the engine launcher
Run this cell — it writes `engine_main.py`, which loads the model and opens the tunnel in a fresh Python process. Expected output: `Writing gt_voice/engine_main.py`.

In [ ]:
%%writefile gt_voice/engine_main.py
"""GOLDTECH Voice engine entrypoint — runs in a fresh Python process."""
import os, sys, time, re, threading, subprocess, urllib.request
from pathlib import Path

BASE = os.environ.get("GT_VOICE_BASE") or os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, BASE)
os.environ.setdefault("HF_HOME", os.path.join(BASE, "hf_cache"))

import torch
import numpy as np
import uvicorn
from chatterbox.tts_turbo import ChatterboxTurboTTS
import server

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Chatterbox Turbo on {device.upper()}… (first run downloads ~1.5 GB)", flush=True)
model = ChatterboxTurboTTS.from_pretrained(device=device)
print(f"✅ Model ready · sample rate {model.sr} Hz", flush=True)

# Force a strict float32 pipeline. Kaggle's torch/pyloudnorm stack can otherwise
# promote audio to float64 and trigger "expected scalar type Float but found Double".
torch.set_default_dtype(torch.float32)
for _m in (model.t3, model.s3gen, model.ve):
    _m.float()
_orig_norm = model.norm_loudness
def _norm_loudness_f32(wav, sr, target_lufs=-27):
    out = _orig_norm(wav, sr, target_lufs)
    try:
        if isinstance(out, np.ndarray):
            return out.astype(np.float32)
        if torch.is_tensor(out):
            return out.to(torch.float32)
    except Exception:
        pass
    return out
model.norm_loudness = _norm_loudness_f32
print("✅ float32 audio pipeline locked", flush=True)

server.MODEL = model
server.DEVICE = device
server.SR = model.sr

PORT = 8000
config = uvicorn.Config(server.app, host="0.0.0.0", port=PORT, log_level="warning")
srv = uvicorn.Server(config)
threading.Thread(target=srv.run, daemon=True).start()

for _ in range(120):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2)
        break
    except Exception:
        time.sleep(0.5)
print("✅ Engine API is up on port", PORT, flush=True)

CF = os.path.join(BASE, "cloudflared")
tunnel = subprocess.Popen(
    [CF, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
public_url = None
start = time.time()
while time.time() - start < 120:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    m = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    Path(BASE, "url.txt").write_text(public_url)
    print("\n" + "=" * 70, flush=True)
    print("✅ GOLDTECH VOICE ENGINE READY", flush=True)
    print("\n   👉  ", public_url, "\n", flush=True)
    print("Paste this URL into the web app under 'Connect engine'.", flush=True)
    print("=" * 70, flush=True)
else:
    print("⚠️ Tunnel URL not detected yet. Check the cloudflared output / engine.log.", flush=True)
    print("   On Kaggle, make sure Internet is ON in Settings.", flush=True)

# Keep the process alive so the API + tunnel stay up.
while True:
    time.sleep(3600)


In [ ]:
# @title 5) Start engine + open tunnel  ← copy the URL it prints
import os, sys, time, subprocess
from pathlib import Path
import os
def _gt_base():
    if os.path.isdir("/kaggle/working"):      # Kaggle
        return "/kaggle/working/gt_voice"
    if os.path.isdir("/content"):             # Google Colab
        return "/content/gt_voice"
    return os.path.abspath("gt_voice")        # Lightning / SageMaker / local
BASE = _gt_base()
os.makedirs(BASE, exist_ok=True)
os.environ["GT_VOICE_BASE"] = BASE
def _gt_platform():
    if os.path.isdir("/kaggle/working"): return "Kaggle"
    if os.path.isdir("/content"): return "Google Colab"
    return "GPU host"

engine_py = Path(BASE) / "engine_main.py"
log_path = Path(BASE) / "engine.log"
url_path = Path(BASE) / "url.txt"
if not engine_py.exists():
    raise RuntimeError("engine_main.py not found — run the previous cell first.")

# Stop a previous engine from an earlier run of this cell, if any.
prev = globals().get("GT_ENGINE_PROC")
if prev is not None and prev.poll() is None:
    prev.terminate()
    try:
        prev.wait(timeout=5)
    except Exception:
        prev.kill()

if url_path.exists():
    url_path.unlink()

env = os.environ.copy()
env["GT_VOICE_BASE"] = BASE
env.setdefault("HF_HOME", str(Path(BASE) / "hf_cache"))

log_f = open(log_path, "w")
proc = subprocess.Popen([sys.executable, str(engine_py)],
                        stdout=log_f, stderr=subprocess.STDOUT, env=env, cwd=BASE)
GT_ENGINE_PROC = proc
print("Starting engine in a fresh Python process…")
print("Log:", log_path)

def _lines():
    try:
        return log_path.read_text(errors="replace").splitlines()
    except Exception:
        return []

seen = 0
start = time.time()
public_url = None
while time.time() - start < 900:   # up to 15 min for the first model download
    if proc.poll() is not None:
        print("\n".join(_lines()[-60:]))
        raise RuntimeError(f"Engine exited early (code {proc.returncode}). See the log above.")
    lines = _lines()
    if len(lines) > seen:
        print("\n".join(lines[seen:]))
        seen = len(lines)
    if url_path.exists():
        public_url = url_path.read_text().strip()
        break
    time.sleep(2)

print("\n" + "=" * 70)
if public_url:
    print("✅ GOLDTECH VOICE ENGINE READY")
    print("\n   👉  ", public_url, "\n")
    print("Paste this URL into the web app under 'Connect engine'.")
else:
    print("⚠️ Timed out waiting for the tunnel URL. Recent log:")
    print("\n".join(_lines()[-40:]))
    print("If the model is still downloading, wait and re-run this cell.")
print("=" * 70)
print("Keep this notebook tab open while generating. To stop, restart the session.")


## Tips for an accurate clone
- **Reference clip:** 15–20 seconds of you speaking naturally, quiet room, no music/echo. Record straight into the web app.
- **Expressiveness:** start at **0.8**. Lower (0.5–0.7) = steadier, more faithful; higher (1.0+) = more dramatic.
- **Chunk size:** ~220 characters keeps the voice consistent on long scripts.
- The engine conditions on your reference **once** and reuses it for every chunk.
- Output is loudness-normalized to −16 LUFS, saved as WAV + MP3.

## Platform notes
- **Kaggle:** ~30 GPU hrs/week. Turn **Internet ON** (Settings) or the tunnel/model download fails. If cloudflared is blocked, use Colab or Lightning instead.
- **Lightning AI Studio:** free monthly credits; persistent storage. Run these cells in a notebook or adapt to a script.
- **SageMaker Studio Lab:** ~4 GPU hrs/day, persistent storage, no credit card.
- **Colab:** free tier disconnects/throttles; re-run the last cell for a new URL.

## Troubleshooting
- **numpy / LlamaModel import error** → this version locks Kaggle's preinstalled numpy and runs the engine in a fresh process. Restart the session, then Run All.
- **"Model is still loading"** → wait for the last cell to finish; first download is ~1.5 GB.
- **409 / a job is already running** → wait or cancel in the app.
- **Tunnel not reachable** → session slept; re-run the last cell, re-paste URL.
- **Out of memory** → restart session, Run All (T4 16 GB is plenty for this 350M model).
- **Engine log:** `gt_voice/engine.log` in the working dir has the full model/tunnel output.
